In [ ]:
import requests, time, datetime, pandas as pd, os
from sqlalchemy import text
from database import get_engine
from get_symbols import get_sp500_symbols_from_web


def get_history_for_symbol(symbol: str, start_date: datetime.date, end_date: datetime.date):
    """
    用 Yahoo Finance chart API 抓整段日線（start_date -> end_date）
    回傳 pandas DataFrame（若失敗回傳 None）
    DataFrame 欄位會是: symbol, data_type, ts, open, high, low, close, volume
    """
    period1 = int(time.mktime(start_date.timetuple()))
    period2 = int(time.mktime((end_date + datetime.timedelta(days=1)).timetuple()))
    url = f"https://query1.finance.yahoo.com/v8/finance/chart/{symbol}"
    params = {
        "period1": period1,
        "period2": period2,
        "interval": "1d",
        "events": "history",
        "includeAdjustedClose": "true",
    }
    headers = {"User-Agent": "Mozilla/5.0"}
    try:
        resp = requests.get(url, params=params, headers=headers, timeout=20)
        resp.raise_for_status()
        data = resp.json()
        if not data.get("chart") or not data["chart"].get("result"):
            return None
        result = data["chart"]["result"][0]
        timestamps = result.get("timestamp", [])
        if not timestamps:
            return None
        quote = result["indicators"]["quote"][0]
        df = pd.DataFrame(
            {
                "symbol": symbol,
                "ts": pd.to_datetime(timestamps, unit="s"),
                "open": quote.get("open"),
                "high": quote.get("high"),
                "low": quote.get("low"),
                "close": quote.get("close"),
                "volume": quote.get("volume"),
            }
        )
        df["data_type"] = "D"
        df = df.dropna(subset=["open", "high", "low", "close"])
        if df.empty:
            return None
        return df[["symbol", "data_type", "ts", "open", "high", "low", "close", "volume"]]
    except Exception as e:
        print(f"history fetch error {symbol}: {e}")
        return None


def insert_daily_bulk(df: pd.DataFrame, engine):
    """
    批次插入到 stock_data，使用 INSERT ... ON CONFLICT DO NOTHING
    逐筆插入示範（若資料量大可改成 COPY）
    """
    if df is None or df.empty:
        return 0
    cols = list(df.columns)
    insert_sql = f"""
    INSERT INTO stock_data ({', '.join(cols)})
    VALUES ({', '.join([f":{c}" for c in cols])})
    ON CONFLICT ON CONSTRAINT uq_symbol_datatype_ts DO NOTHING
    """
    inserted = 0
    with engine.begin() as conn:
        for row in df.to_dict(orient="records"):
            try:
                conn.execute(text(insert_sql), row)
                inserted += 1
            except Exception as e:
                print(f"insert row error {row.get('symbol')} {row.get('ts')}: {e}")
    return inserted


def initial_history_all(start_date: datetime.date, batch_size=50, per_symbol_sleep=1.0, batch_sleep=30):
    """
    Batch history ingestion with progress tracking.
    """
    end_date = datetime.date.today() - datetime.timedelta(days=1)
    symbols = get_sp500_symbols_from_web()
    engine = get_engine()
    total = 0
    prog_file = "history_progress.txt"
    done = set()
    if os.path.exists(prog_file):
        with open(prog_file, "r") as f:
            for line in f:
                done.add(line.strip())

    for i in range(0, len(symbols), batch_size):
        batch = symbols[i : i + batch_size]
        for s in batch:
            if s in done:
                print(f"{s} already done, skip")
                continue
            try:
                df = get_history_for_symbol(s, start_date, end_date)
                if df is None or df.empty:
                    print(f"{s} history empty, skip")
                else:
                    n = insert_daily_bulk(df, engine)
                    total += n
                    print(f"{s} inserted {n} rows")
                with open(prog_file, "a") as f:
                    f.write(s + "\n")
                done.add(s)
            except Exception as e:
                print(f"Error processing {s}: {e}")
            time.sleep(per_symbol_sleep)
        print(f"Batch {i//batch_size + 1} done, sleeping {batch_sleep}s")
        time.sleep(batch_sleep)

    print("initial history completed, total rows:", total)
    return total


if __name__ == "__main__":
    start = datetime.date(2015, 1, 1)
    initial_history_all(start)